In [16]:
import cv2
import numpy as np
import json
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from get_court_position_methods import calibrate_camera, get_3d_position, pts_real_3d, pts_video_2d
from IMM_UKF import UnscentedKalmanFilter, IMMEstimator, unscented_transform
from filterpy.kalman import MerweScaledSigmaPoints
from filterpy.common import Q_discrete_white_noise
from scipy.linalg import block_diag, cholesky
from scipy.stats import multivariate_normal

from ultralytics import YOLO
import albumentations as A
import torch
import torchvision
import torchaudio

In [2]:
def fx_ballistic(x, dt):
    """Балістична нелінійна модель: x_next = F*x * gravity_effect
    F-матриця для [x, vx, y, vy, z, vz]"""
    vx, vy, vz = x[1], x[3], x[5]
    F = np.array([[1, dt, 0, 0,  0,  0],                  # x
                  [0, 1,  0, 0,  0,  0],                  # vx
                  [0, 0,  1, dt, 0,  0],                  # y
                  [0, 0,  0, 1,  0,  0],                  # vy
                  [0, 0,  0, 0,  1,  dt],                 # z
                  [0, 0,  0, 0,  0,  1]], dtype=float)    # vz

    B = np.diag([0.5 * dt**2, dt, 0.5 * dt**2, dt, 0.5 * dt**2, dt])
    k = 0.0314
    v_mag = np.sqrt(x[1]**2 + x[3]**2 + x[5]**2)
    a_dx = -k * v_mag * vx
    a_dy = -k * v_mag * vy
    a_dz = -k * v_mag * vz
    u = np.array([a_dx, a_dx, a_dy - 9.81, a_dy - 9.81, a_dz, a_dz])

    x_next = np.dot(F, x) + np.dot(B, u)

    return x_next

def fx_hit(x, dt):
    """
    Модель удару: Constant Velocity.
    Стан: [x, vx, y, vy, z, vz]
    """
    F = np.array([[1, dt, 0, 0,  0,  0],
                  [0, 1,  0, 0,  0,  0],
                  [0, 0,  1, dt, 0,  0],
                  [0, 0,  0, 1,  0,  0],
                  [0, 0,  0, 0,  1,  dt],
                  [0, 0,  0, 0,  0,  1]], dtype=float)

    return np.dot(F, x)

def fx_bounce(x, dt):
    """Модель відскоку, марковська матриця"""
    espilon = 0.75 # Коефіцієнт реституції
    friction = 0.85 # Коефіцієнт тертя
    x_next = np.copy(x)
    vx_new = x[1] * friction
    vy_new = -x[3] * epsilon
    vz_new = x[5] * friction

    x_next[0] += vx_new * dt
    x_next[2] += xy_new * dt
    x_next[4] += vz_new * dt

    x_next[1] = vx_new
    x_next[3] = vy_new
    x_next[5] = vz_new

    return x_next

def hx(x):
    return np.array([x[0], x[2], x[4]])

def measurement_transform(prediction_data, K, R_matrix, camera_pos, ball_diameter=0.21):
    u_cam, v_cam, w_cam = prediction_data['x_pos'], prediction_data['y_pos'], prediction_data['w_box']
    raw_x, raw_y, raw_z = get_3d_position(u_cam, v_cam, w_cam, K, R_matrix, camera_pos, ball_diameter)
    measurement = np.array([raw_x, raw_y, raw_z], dtype=np.float32)

    return measurement

def get_dynamic_transition_matrix(y, vy):
    """
    Повертає марковську матрицю 3x3 для моделей: [Ballistic, Hit, Bounce]
    """
    M = np.array([[0.95, 0.04, 0.01],
                  [0.60, 0.40, 0.00],
                  [0.90, 0.00, 0.10]])

    if y < 0.3 and vy < 0:
        M[0] = [0.10, 0.05, 0.85]
        M[1] = [0.10, 0.05, 0.85]

    return M

In [3]:

frames_per_second = 50
dt = 1 / frames_per_second
mu = np.array([0.95, 0.04, 0.01])

# Для балістичної моделі Q мінімальна
q_var_ballistic = 0.1
q_b = Q_discrete_white_noise(dim=2, dt=dt, var=q_var_ballistic)
Q_ballistic = block_diag(q_b, q_b, q_b)

# Для моделі удару Q велика
q_var_hit = 100
q_h = Q_discrete_white_noise(dim=2, dt=dt, var=q_var_hit)
Q_hit = block_diag(q_h, q_h, q_h)

# Для відскоку Q середня
q_var_bounce = 5
q_bnc = Q_discrete_white_noise(dim=2, dt=dt, var=q_var_bounce)
Q_bounce = block_diag(q_bnc, q_bnc, q_bnc)

In [4]:
path_to_load = 'detect_infos/detection_data.json'
with open(path_to_load, 'r', encoding='utf-8') as f:
    raw_detections = json.load(f)

detections_df = pd.DataFrame.from_dict(raw_detections)
detections_df.set_index('frame', inplace=True)

In [5]:
# main_model = YOLO('/home/var-roman/Desktop/my_projects/diploma_project/training_models/models/main_model_april.pt')
#
# results = main_model.predict(
#     source='/home/var-roman/Desktop/my_projects/diploma_project/data/videos/Japan_vs_Poland_ultrashort.mp4',
#     imgsz=1920,
#     conf=0.4,
#     iou=0.45,
#     save=False,
#     stream=True
# )
#
# raw_detections = []
# frame_count = 0
#
# for r in results:
#     frame_count += 1
#
#     if len(r.boxes) > 0:
#         box = r.boxes[0].xywh[0]
#         raw_detections.append({
#             'ball_detected': True,
#             'frame': frame_count,
#             'x_pos': float(box[0]),
#             'y_pos': float(box[1]),
#             'w_box': float(box[2]),
#         })
#     else:
#         raw_detections.append({'ball_detected': False, 'frame': frame_count})
#
# path_to_save = 'detect_infos/raw_detection_data.json'
# with open(path_to_save, 'w', encoding='utf-8') as f:
#     json.dump(raw_detections, f, ensure_ascii=False, indent=4)

In [6]:
detections_df

,ball_detected,x_pos,y_pos,w_box
frame,,,,
1,False,NaN,NaN,NaN
2,False,NaN,NaN,NaN
3,False,NaN,NaN,NaN
4,False,NaN,NaN,NaN
5,False,NaN,NaN,NaN
...,...,...,...,...
1131,False,NaN,NaN,NaN
1132,False,NaN,NaN,NaN
1133,False,NaN,NaN,NaN


In [7]:
dim_x = 6
dim_z = 1
pts_real_3d = np.array([
    [0.0, 0.0, 0.0],
    [9.0, 0.0, 0.0],
    [9.0, 18.0, 0.0],
    [0.0, 18.0, 0.0]],dtype=np.float32)

pts_video_2d = np.array([
    [73, 1031],
    [1835, 1027],
    [1504, 606],
    [405, 606]],dtype=np.float32)

K = np.array([
    [1300.0, 0.0,    960.0],
    [0.0,    1300.0, 540.0],
    [0.0,    0.0,    1.0]],dtype=np.float32)
points = MerweScaledSigmaPoints(n=dim_x, alpha=.1, beta=2., kappa=1.)
R, tvec, camera_pos = calibrate_camera(pts_real_3d, pts_video_2d, K, dist=np.zeros((4, 1)))

UKF = UnscentedKalmanFilter(dim_x=dim_x, dim_z=dim_z, dt=dt, fx=fx_ballistic, hx=hx, points=points)
UKF.P = np.diag([0.1, 50.0, 0.1, 50.0, 0.1, 50.0])
UKF.Q = Q_ballistic
UKF.R = np.diag([0.01, 0.01, 0.04])

In [9]:
is_initialized = False
smoothed_x, smoothed_y, smoothed_z = [], [], []

for i in detections_df.iterrows():
    if not is_initialized:
        if i[1]['ball_detected']:
            z = measurement_transform(i[1], K, R, camera_pos)
            UKF.x = np.zeros([6])
            UKF.x[0], UKF.x[2], UKF.x[4] = z
            is_initialized = True
        else:
            continue

    UKF.predict()

    if i[1]['ball_detected']:
        z = measurement_transform(i[1], K, R, camera_pos)
        UKF.update(z)
    else:
        UKF.update(None)

    smoothed_x.append(UKF.x[0]), smoothed_y.append(UKF.x[2]), smoothed_z.append(UKF.x[4])

[np.float64(1.9419959563544658),
 np.float64(1.9219132832074477),
 np.float64(1.9101843218853012),
 np.float64(1.904315166679705),
 np.float64(1.8935704428234204),
 np.float64(1.8927693103179273),
 np.float64(1.850191328468476),
 np.float64(1.8486684678108976),
 np.float64(1.8430997962073998),
 np.float64(1.8279097085085818),
 np.float64(1.8094518018814219),
 np.float64(1.8073378339063453),
 np.float64(1.8052265613887268),
 np.float64(1.8031182371079435),
 np.float64(1.8010131121986692),
 np.float64(1.7989114359869909),
 np.float64(1.796813455833897),
 np.float64(1.7947194169848473),
 np.float64(1.7926295624249544),
 np.float64(1.7905441327388143),
 np.float64(1.7884633659748523),
 np.float64(1.786387497514303),
 np.float64(1.7843167599444065),
 np.float64(1.782251382936117),
 np.float64(1.7801915931258634),
 np.float64(1.7781376140022633),
 np.float64(1.776089665796981),
 np.float64(1.7740479653801025),
 np.float64(1.7720127261602965),
 np.float64(1.7699841579895637),
 np.float64(1.76

In [15]:
fig = px.line_3d(x=smoothed_x, y=smoothed_y, z=smoothed_z, width=500, height=500)
fig.show()